In [1]:
# Install required packages (uncomment as needed)
!pip install transformers datasets scikit-learn torch evaluate accelerate
!pip install sentencepiece

# Core imports
import random
import numpy as np
import pandas as pd

# HuggingFace
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, pipeline
import evaluate

# Scikit-learn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.5 MB/s eta 0:00:00


In [2]:
import torch
torch.cuda.is_available()

True

In [3]:
# TODO: Load your assigned dataset

dataset = load_dataset("csv",
    data_files={
        "train": "train.csv",
        "validation": "val.csv",
        "test": "test_input.csv"
    }
)

print(dataset)

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['ID', 'Post', 'Comment', 'Label'],
        num_rows: 720
    })
    validation: Dataset({
        features: ['ID', 'Post', 'Comment', 'Label'],
        num_rows: 240
    })
    test: Dataset({
        features: ['ID', 'Post', 'Comment', 'Label'],
        num_rows: 240
    })
})


In [4]:
# label encoding
# encode the "name" column of the train and val datasets into a numerical column "label"
labels = list(dataset["train"].unique("Label"))
label_to_id = {l: i for i, l in enumerate(labels)}
id_to_label = {i: l for l, i in label_to_id.items()}

def encode_labels(batch):
    batch["label"] = [label_to_id[name] for name in batch["Label"]]
    return batch

# apply to train and val datasets
train = dataset["train"].map(encode_labels, batched=True)
val = dataset["validation"].map(encode_labels, batched=True)
test = dataset["test"]  # no labels here

Map:   0%|          | 0/720 [00:00<?, ? examples/s]

Map:   0%|          | 0/240 [00:00<?, ? examples/s]

In [5]:
MODEL_NAME = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(labels),
    id2label=id_to_label,
    label2id=label_to_id
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [6]:
def tokenize(data):
    return tokenizer(data["Post"], data["Comment"], truncation=True, padding="max_length", max_length=128)

train_tokenized = train.map(tokenize, batched=True)
val_tokenized = val.map(tokenize, batched=True)
test_tokenized = test.map(tokenize, batched=True)

Map:   0%|          | 0/720 [00:00<?, ? examples/s]

Map:   0%|          | 0/240 [00:00<?, ? examples/s]

Map:   0%|          | 0/240 [00:00<?, ? examples/s]

In [7]:
print(train_tokenized[0])

{'ID': 163.0, 'Post': '"Is there a politician you strongly disagree with politically but still respect? Why?"', 'Comment': '"Mamdani. Far too left for me but good personality, genuine and impressive victory."', 'Label': 'direct', 'label': 0, 'input_ids': [101, 1000, 2003, 2045, 1037, 3761, 2017, 6118, 21090, 2007, 10317, 2021, 2145, 4847, 1029, 2339, 1029, 1000, 102, 1000, 5003, 26876, 7088, 1012, 2521, 2205, 2187, 2005, 2033, 2021, 2204, 6180, 1010, 10218, 1998, 8052, 3377, 1012, 1000, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [9]:
# TODO: Define training arguments and Trainer, then fine-tune Model 1
training_args = TrainingArguments(output_dir="./results",
                                  per_device_train_batch_size=8,
                                  per_device_eval_batch_size=8,
                                  num_train_epochs=6,
                                  learning_rate=3e-5,
                                  eval_strategy="epoch",
                                  save_strategy="epoch",
                                  weight_decay=0.01,
                                  max_grad_norm=1.0,
                                  lr_scheduler_type="linear",
                                  seed=SEED,
                                  load_best_model_at_end=True,
                                  metric_for_best_model="f1_macro",
                                  greater_is_better=True,
                                  fp16=True,
                                  dataloader_num_workers=2)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro")
    }

trainer = Trainer(model=model,
                    args=training_args,
                    train_dataset=train_tokenized,
                    eval_dataset=val_tokenized,
                    compute_metrics=compute_metrics)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,No log,0.781848,0.587500,0.388344
2,No log,0.627445,0.733333,0.576646
3,No log,0.830715,0.725000,0.596260
4,No log,0.942967,0.766667,0.656895
5,No log,1.086244,0.737500,0.647531
6,0.279002,1.125859,0.750000,0.656929


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=540, training_loss=0.2610892454783122, metrics={'train_runtime': 102.2886, 'train_samples_per_second': 42.233, 'train_steps_per_second': 5.279, 'total_flos': 284162491146240.0, 'train_loss': 0.2610892454783122, 'epoch': 6.0})

In [10]:
# test predictions
test_preds = trainer.predict(test_tokenized)
test_names = np.argmax(test_preds.predictions, axis=1)

test_labels = [id_to_label[i] for i in test_names]
test_ids = dataset["test"]["ID"]

submission = pd.DataFrame({"ID": test_ids, "Label": test_labels})
submission.to_csv("predictions.csv", index=False)